In [1]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
# 🔒 Disable W&B
os.environ["WANDB_DISABLED"] = "true"

In [3]:
# 📥 Load dataset
df = pd.read_csv('./product_reviews.csv')  # Make sure this file is in the same directory

In [4]:
# 🧹 Clean and encode
df = df[['review', 'label']].dropna()
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])

In [5]:
# 🔀 Split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['review'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

In [6]:
# 🔤 Tokenization
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

In [7]:
# 🧱 Dataset wrapper
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset = SentimentDataset(val_encodings, val_labels)

In [8]:
# 🧠 Load model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# ⚙️ Training config
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch"

)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [11]:

# 🧰 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

# 🚀 Train
trainer.train()

# 💾 Save final model
trainer.save_model('./bert-finetuned-final')
print("✅ Model saved to './bert-finetuned-final'")

# 🔍 Inference
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([predicted_class])[0]


C:\Users\91739\AppData\Local\Temp\ipykernel_32044\1457426187.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.472700,0.278737
2,0.138200,0.140347
3,0.059800,0.174773


c:\Users\91739\OneDrive\Desktop\FINE_TUNING\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\91739\OneDrive\Desktop\FINE_TUNING\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ Model saved to './bert-finetuned-final'


In [ ]:
from ipywidgets import Text, Button, VBox, Output
from IPython.display import display

out = Output()

text_box = Text(
    value='',
    placeholder='Type your product review here...',
    description='Review:',
    disabled=False
)

button = Button(description="Predict Sentiment")

def on_button_clicked(b):
    review = text_box.value
    sentiment = predict_sentiment(review)
    with out:
        out.clear_output()
        print(f"Predicted sentiment: {sentiment}")

button.on_click(on_button_clicked)

display(VBox([text_box, button, out]))
